In [0]:
%sql
CREATE OR REPLACE TABLE automobilerepair.gold.cube_kpi_metrics AS
SELECT 
    -- Primary Keys
    o.orderlifecycle_id,
    o.order_id,
    
    -- Store Dimensions
    o.store_id,
    s.store_name,
    s.manager_id,
    s.manager_name,
    s.store_type,
    
    -- Technician Dimensions
    o.technician_id,
    t.technician_name,
    
    -- Service Information
    o.service_type,
    o.order_status,
    
    -- Vehicle Information
    o.vehicle_no,
    
    -- Date/Time Fields
    o.vehicle_in_datetime,
    o.vehicle_out_datetime,
    o.actual_work_start_datetime,
    o.actual_completion_datetime,
    o.promised_delivery_datetime,
    o.actual_delivery_datetime,
    o.invoice_date,
    
    -- Financial Fields
    o.invoice_id,
    o.invoice_amount,
    o.estimate_id,
    o.estimate_amount,
    b.budget_amount,
    
    -- Estimator Dimensions
    e.estimator_id,
    e.estimator_name,
    e.estimate_type,
    o.version_no as estimate_version_no,

    -- Survey Fields
    cs.survey_id,
    cs.responded_flag AS survey_responded_flag,
    cs.delivered_on_time_rating,
    cs.work_quality_rating,
    cs.cleanliness_rating,
    cs.communication_rating,
    
    -- Calculated Metrics: Time Durations (in days)
    DATEDIFF(DAY, o.vehicle_in_datetime, o.vehicle_out_datetime) AS days_in_shop,
    DATEDIFF(DAY, o.vehicle_in_datetime, o.actual_work_start_datetime) AS days_vehicle_in_to_work_start,
    DATEDIFF(DAY, o.actual_work_start_datetime, o.actual_completion_datetime) AS days_work_start_to_completion,
    DATEDIFF(DAY, o.actual_completion_datetime, o.actual_delivery_datetime) AS days_completion_to_delivery,
    
    -- Calculated Metrics: Completion Time Accuracy (in days, negative means early, positive means late)
    DATEDIFF(DAY, o.promised_delivery_datetime, o.actual_delivery_datetime) AS completion_time_variance_days,
    ABS(DATEDIFF(DAY, o.promised_delivery_datetime, o.actual_delivery_datetime)) AS completion_time_accuracy_abs_days,
    
    -- Calculated Metrics: Estimate Accuracy
    CASE 
        WHEN o.estimate_amount > 0 THEN 
            ABS(o.invoice_amount - o.estimate_amount) * 100.0 / o.estimate_amount 
        ELSE NULL 
    END AS estimate_variance_pct,
    
    -- Calculated Metrics: Survey Overall Satisfaction (average of all ratings)
    CASE 
        WHEN cs.responded_flag = true THEN 
            (COALESCE(cs.delivered_on_time_rating, 0) + 
             COALESCE(cs.work_quality_rating, 0) + 
             COALESCE(cs.cleanliness_rating, 0) + 
             COALESCE(cs.communication_rating, 0)) / 4.0
        ELSE NULL 
    END AS overall_satisfaction_rating,
    
    -- Date Dimensions for Time-based Analysis
    DATE_TRUNC('month', o.invoice_date) AS invoice_month,
    YEAR(o.invoice_date) AS invoice_year,
    MONTH(o.invoice_date) AS invoice_month_num,
    DATE_FORMAT(o.invoice_date, 'yyyy-MM') AS invoice_year_month,
    
    -- Flags for Filtering
    CASE WHEN o.order_status = 'COMPLETED' THEN 1 ELSE 0 END AS is_completed,
    CASE WHEN o.invoice_amount IS NOT NULL THEN 1 ELSE 0 END AS has_invoice
    
FROM 
    automobilerepair.gold.fact_orderlifecycle o
    
    -- Join Store Information
    LEFT JOIN automobilerepair.gold.dim_store s 
        ON o.store_id = s.store_id
    
    -- Join Technician Information
    LEFT JOIN automobilerepair.gold.dim_technician t 
        ON o.technician_id = t.technician_id
    
    -- Join Survey Information
    LEFT JOIN automobilerepair.gold.dim_customer_survey cs 
        ON o.order_id = cs.order_id
    
    -- Join Estimate Information
    LEFT JOIN automobilerepair.gold.dim_estimate e 
        ON o.estimate_id = e.estimate_id
    
    -- Join Budget Information (matching on store and month)
    LEFT JOIN automobilerepair.gold.fact_budget b 
        ON o.store_id = b.store_id 
        AND DATE_FORMAT(o.invoice_date, 'yyyy-MM') = b.month

In [0]:
row_count = spark.table("automobilerepair.gold.cube_kpi_metrics").count()
print(row_count)

###  MTD Performance vs Previous MTD

In [0]:
%sql
-- KPI 1: MTD Performance vs Previous MTD
-- Month-to-date revenue and completed orders compared with previous month-to-date

WITH latest_data AS (
    -- Get the latest invoice date available in the cube
    SELECT MAX(invoice_date) AS max_invoice_date
    FROM automobilerepair.gold.cube_kpi_metrics
),

current_mtd AS (
    SELECT
        store_id,
        store_name,
        manager_id,
        manager_name,
        SUM(invoice_amount) AS current_mtd_revenue,
        COUNT(CASE WHEN is_completed = 1 THEN 1 END) AS current_mtd_orders,
        ld.max_invoice_date AS reference_date,
        DAY(ld.max_invoice_date) AS days_into_month
    FROM automobilerepair.gold.cube_kpi_metrics
    CROSS JOIN latest_data ld
    WHERE invoice_date >= DATE_TRUNC('MONTH', ld.max_invoice_date)   -- 1st of current month
      AND invoice_date <= ld.max_invoice_date                       -- up to latest available date
      AND has_invoice = 1
    GROUP BY store_id, store_name, manager_id, manager_name, ld.max_invoice_date
),

previous_mtd AS (
    SELECT
        store_id,
        SUM(invoice_amount) AS previous_mtd_revenue,
        COUNT(CASE WHEN is_completed = 1 THEN 1 END) AS previous_mtd_orders
    FROM automobilerepair.gold.cube_kpi_metrics
    CROSS JOIN latest_data ld
    WHERE invoice_date >= DATE_TRUNC('MONTH', ADD_MONTHS(ld.max_invoice_date, -1))   -- 1st of previous month
      AND invoice_date <= DATE_ADD(
                            DATE_TRUNC('MONTH', ADD_MONTHS(ld.max_invoice_date, -1)), 
                            DAY(ld.max_invoice_date) - 1
                          )                                                          -- same day-of-month
      AND has_invoice = 1
    GROUP BY store_id
)

SELECT
    c.store_id,
    c.store_name,
    c.manager_id,
    c.manager_name,
    
    c.current_mtd_revenue,
    COALESCE(p.previous_mtd_revenue, 0) AS previous_mtd_revenue,
    c.current_mtd_revenue - COALESCE(p.previous_mtd_revenue, 0) AS revenue_variance,
    ROUND(
        (c.current_mtd_revenue - COALESCE(p.previous_mtd_revenue, 0)) * 100.0 
        / NULLIF(p.previous_mtd_revenue, 0), 
        2
    ) AS revenue_variance_pct,

    c.current_mtd_orders,
    COALESCE(p.previous_mtd_orders, 0) AS previous_mtd_orders,
    c.current_mtd_orders - COALESCE(p.previous_mtd_orders, 0) AS orders_variance,
    ROUND(
        (c.current_mtd_orders - COALESCE(p.previous_mtd_orders, 0)) * 100.0 
        / NULLIF(p.previous_mtd_orders, 0), 
        2
    ) AS orders_variance_pct,

    c.days_into_month,
    c.reference_date,
    CURRENT_TIMESTAMP() AS created_at

FROM current_mtd c
LEFT JOIN previous_mtd p ON c.store_id = p.store_id
ORDER BY revenue_variance_pct DESC NULLS LAST;

In [0]:
%sql
SELECT COUNT(*) AS total_rows,
       MIN(invoice_date) AS earliest_date,
       MAX(invoice_date) AS latest_date
FROM automobilerepair.gold.cube_kpi_metrics;

In [0]:
%sql
SELECT 
    COUNT(*) AS current_mtd_rows,
    MIN(invoice_date) AS first_invoice_this_month,
    MAX(invoice_date) AS last_invoice_this_month,
    COUNT(CASE WHEN has_invoice = 1 THEN 1 END) AS has_invoice_count
FROM automobilerepair.gold.cube_kpi_metrics
WHERE invoice_date >= DATE_TRUNC('MONTH', CURRENT_DATE());

In [0]:
%sql
SELECT 
    COUNT(*) AS previous_mtd_rows,
    MIN(invoice_date) AS first_invoice_prev_month,
    MAX(invoice_date) AS last_invoice_prev_month
FROM automobilerepair.gold.cube_kpi_metrics
WHERE invoice_date >= DATE_TRUNC('MONTH', ADD_MONTHS(CURRENT_DATE(), -1))
  AND invoice_date < DATE_ADD(DATE_TRUNC('MONTH', ADD_MONTHS(CURRENT_DATE(), -1)), DAY(CURRENT_DATE()) - 1);

In [0]:
%sql
-- Minimal test - just current MTD
SELECT 
    store_id,
    store_name,
    manager_name,
    SUM(invoice_amount) AS current_revenue,
    COUNT(*) AS total_orders
FROM automobilerepair.gold.cube_kpi_metrics
WHERE invoice_date >= DATE_TRUNC('MONTH', CURRENT_DATE())
  AND has_invoice = 1
GROUP BY store_id, store_name, manager_name
ORDER BY current_revenue DESC;

### Average Days in Shop

In [0]:
%sql
-- KPI 2: Average Days in Shop
-- Average days a vehicle spends in the shop, by store and service type

SELECT 
    store_name,
    store_type,
    manager_name,
    service_type,
    COUNT(*) AS total_orders,
    ROUND(AVG(days_in_shop), 2) AS avg_days_in_shop,
    MIN(days_in_shop) AS min_days_in_shop,
    MAX(days_in_shop) AS max_days_in_shop
FROM automobilerepair.gold.cube_kpi_metrics
WHERE 
    vehicle_in_datetime IS NOT NULL 
    AND vehicle_out_datetime IS NOT NULL
    AND days_in_shop IS NOT NULL
    AND days_in_shop >= 0
GROUP BY store_name, store_type, manager_name, service_type
ORDER BY avg_days_in_shop DESC

### Survey Coverage

In [0]:
%sql
-- KPI 3: Survey Coverage
-- Number of surveys sent versus number of surveys responded, by store

SELECT 
    store_name,
    store_type,
    manager_name,
    COUNT(DISTINCT survey_id) AS surveys_sent,
    SUM(CASE WHEN survey_responded_flag = true THEN 1 ELSE 0 END) AS surveys_responded,
    COUNT(DISTINCT survey_id) - SUM(CASE WHEN survey_responded_flag = true THEN 1 ELSE 0 END) AS surveys_not_responded,
    ROUND(SUM(CASE WHEN survey_responded_flag = true THEN 1 ELSE 0 END) * 100.0 / COUNT(DISTINCT survey_id), 2) AS response_rate_pct
FROM automobilerepair.gold.cube_kpi_metrics
WHERE survey_id IS NOT NULL
GROUP BY store_name, store_type, manager_name
ORDER BY response_rate_pct DESC

### Survey Scores Summary

In [0]:
%sql

SELECT 
    store_name,
    store_type,
    manager_name,
    COUNT(*) AS total_surveys_responded,
    ROUND(AVG(delivered_on_time_rating), 2) AS avg_delivered_on_time,
    ROUND(AVG(work_quality_rating), 2) AS avg_work_quality,
    ROUND(AVG(cleanliness_rating), 2) AS avg_cleanliness,
    ROUND(AVG(communication_rating), 2) AS avg_communication,
    ROUND(AVG(overall_satisfaction_rating), 2) AS overall_satisfaction,
    RANK() OVER (ORDER BY AVG(overall_satisfaction_rating) DESC) AS satisfaction_rank
FROM automobilerepair.gold.cube_kpi_metrics
WHERE survey_responded_flag = true AND overall_satisfaction_rating IS NOT NULL
GROUP BY store_name, store_type, manager_name
ORDER BY satisfaction_rank ASC

### Revenue vs Budget

In [0]:
%sql

SELECT 
    manager_id,
    manager_name,
    store_name,
    invoice_year_month,
    SUM(invoice_amount) AS actual_revenue,
    MAX(budget_amount) AS budget_amount,
    SUM(invoice_amount) - MAX(budget_amount) AS revenue_variance,
    CASE 
        WHEN MAX(budget_amount) > 0 THEN 
            ROUND(SUM(invoice_amount) * 100.0 / MAX(budget_amount), 2)
        ELSE NULL 
    END AS budget_achievement_pct,
    RANK() OVER (
        PARTITION BY invoice_year_month 
        ORDER BY 
            CASE 
                WHEN MAX(budget_amount) > 0 THEN SUM(invoice_amount) * 100.0 / MAX(budget_amount)
                ELSE 0 
            END DESC
    ) AS achievement_rank
FROM 
    automobilerepair.gold.cube_kpi_metrics
WHERE has_invoice = 1
    AND invoice_year_month IS NOT NULL
    AND budget_amount IS NOT NULL
GROUP BY manager_id, manager_name, store_name, invoice_year_month
ORDER BY invoice_year_month DESC, achievement_rank ASC

In [0]:
%sql
WITH monthly_revenue AS (
    SELECT 
        manager_id,
        manager_name,
        DATE_TRUNC('MONTH', invoice_date) AS month_start,
        SUM(invoice_amount) AS actual_revenue,
        COUNT(DISTINCT order_id) AS total_orders
        
    FROM automobilerepair.gold.cube_kpi_metrics
    WHERE invoice_amount IS NOT NULL 
      AND has_invoice = 1
      AND manager_id IS NOT NULL
    GROUP BY manager_id, manager_name, DATE_TRUNC('MONTH', invoice_date)
),

budget_comparison AS (
    SELECT 
        r.manager_id,
        r.manager_name,
        r.month_start,
        r.actual_revenue,
        r.total_orders,
        
        COALESCE(b.budget_amount, 0) AS budgeted_revenue,
        r.actual_revenue - COALESCE(b.budget_amount, 0) AS revenue_variance,
        
        ROUND(
            CASE 
                WHEN COALESCE(b.budget_amount, 0) > 0 
                THEN (r.actual_revenue - b.budget_amount) * 100.0 / b.budget_amount 
                ELSE NULL 
            END, 2
        ) AS budget_achievement_pct
        
    FROM monthly_revenue r
    LEFT JOIN automobilerepair.gold.cube_kpi_metrics b 
      ON r.manager_id = b.manager_id 
     AND r.month_start = DATE_TRUNC('MONTH', b.invoice_date)
     AND b.budget_amount IS NOT NULL  
)

SELECT 
    manager_id,
    manager_name,
    month_start,
    actual_revenue,
    budgeted_revenue,
    revenue_variance,
    budget_achievement_pct,
    
    -- Ranking: Highest budget achievement % gets Rank 1
    RANK() OVER (PARTITION BY month_start 
                 ORDER BY budget_achievement_pct DESC) AS budget_rank,
    
    CURRENT_TIMESTAMP() AS created_at
FROM budget_comparison
ORDER BY month_start DESC, budget_rank;

### Top Technicians by Completion Time Accuracy

In [0]:
%sql


WITH technician_accuracy AS (
    SELECT 
        store_name,
        store_id,
        technician_id,
        technician_name,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(AVG(completion_time_accuracy_abs_days), 2) AS avg_accuracy_days,
        SUM(CASE WHEN completion_time_variance_days = 0 THEN 1 ELSE 0 END) AS on_time_deliveries,
        SUM(CASE WHEN completion_time_variance_days < 0 THEN 1 ELSE 0 END) AS early_deliveries,
        SUM(CASE WHEN completion_time_variance_days > 0 THEN 1 ELSE 0 END) AS late_deliveries,
        ROUND(SUM(CASE WHEN completion_time_variance_days = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS on_time_pct
    FROM 
        automobilerepair.gold.cube_kpi_metrics
    WHERE 
        promised_delivery_datetime IS NOT NULL
        AND actual_delivery_datetime IS NOT NULL
        AND completion_time_accuracy_abs_days IS NOT NULL
        AND is_completed = 1
    GROUP BY store_name, store_id, technician_id, technician_name
      -- Only include technicians with at least 5 orders
),
ranked_technicians AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY store_id ORDER BY avg_accuracy_days ASC, on_time_pct DESC) AS rank_in_store
    FROM technician_accuracy
)
SELECT 
    store_id,
    store_name,
    technician_name,
    total_orders,
    on_time_deliveries,
    early_deliveries,
    late_deliveries,
    on_time_pct,
    rank_in_store
FROM ranked_technicians
WHERE rank_in_store <= 5
ORDER BY store_id, rank_in_store

### Year-to-Date Revenue Growth

In [0]:
%sql
WITH latest_data AS (
    -- Get the latest invoice date available in the data
    SELECT MAX(invoice_date) AS max_invoice_date
    FROM automobilerepair.gold.cube_kpi_metrics
),

current_ytd AS (
    SELECT
        store_id,
        store_name,
        SUM(invoice_amount) AS current_ytd_revenue,
        ld.max_invoice_date AS reference_date,
        YEAR(ld.max_invoice_date) AS current_year,
        DAYOFYEAR(ld.max_invoice_date) AS day_of_year
    FROM automobilerepair.gold.cube_kpi_metrics
    CROSS JOIN latest_data ld
    WHERE invoice_date >= DATE_TRUNC('YEAR', ld.max_invoice_date)   -- 1st Jan current year
      AND invoice_date <= ld.max_invoice_date                       -- up to latest data
      AND has_invoice = 1
    GROUP BY store_id, store_name, ld.max_invoice_date
),

previous_ytd AS (
    SELECT
        store_id,
        SUM(invoice_amount) AS previous_ytd_revenue
    FROM automobilerepair.gold.cube_kpi_metrics
    CROSS JOIN latest_data ld
    WHERE invoice_date >= DATE_TRUNC('YEAR', ADD_MONTHS(ld.max_invoice_date, -12))   -- 1st Jan previous year
      AND invoice_date <= DATE_ADD( DATE_TRUNC('YEAR', ADD_MONTHS(ld.max_invoice_date, -12)), DAYOFYEAR(ld.max_invoice_date) - 1)
      AND has_invoice = 1
    GROUP BY store_id
),

growth_calc AS (
    SELECT
        c.store_id,
        c.store_name,
        c.current_ytd_revenue,
        COALESCE(p.previous_ytd_revenue, 0) AS previous_ytd_revenue,
        c.current_ytd_revenue - COALESCE(p.previous_ytd_revenue, 0) AS revenue_variance,
        ROUND(
            (c.current_ytd_revenue - COALESCE(p.previous_ytd_revenue, 0)) * 100.0 
            / NULLIF(p.previous_ytd_revenue, 0), 
            2
        ) AS ytd_growth_pct,
        c.reference_date,
        c.current_year,
        c.day_of_year
    FROM current_ytd c
    LEFT JOIN previous_ytd p ON c.store_id = p.store_id
)

SELECT
    store_id,
    store_name,
    current_ytd_revenue,
    previous_ytd_revenue,
    revenue_variance,
    ytd_growth_pct,
    RANK() OVER (ORDER BY ytd_growth_pct DESC) AS growth_rank,
    date(reference_date) as reference_date
FROM growth_calc
ORDER BY growth_rank;

### Stage-wise Day Cycle Time

In [0]:
%sql

SELECT 
    store_id,
    store_name,
    store_type,
    manager_name,
    service_type,
    COUNT(*) AS total_orders,
    ROUND(AVG(days_vehicle_in_to_work_start), 2) AS avg_days_vehicle_in_to_work_start,
    ROUND(AVG(days_work_start_to_completion), 2) AS avg_days_work_start_to_completion,
    ROUND(AVG(days_completion_to_delivery), 2) AS avg_days_completion_to_delivery,
    ROUND(AVG(days_in_shop), 2) AS avg_total_days_in_shop,
    ROUND(AVG(days_vehicle_in_to_work_start + days_work_start_to_completion + days_completion_to_delivery), 2) AS avg_total_cycle_time
FROM automobilerepair.gold.cube_kpi_metrics
WHERE days_vehicle_in_to_work_start IS NOT NULL
    AND days_work_start_to_completion IS NOT NULL
    AND days_completion_to_delivery IS NOT NULL
    AND days_vehicle_in_to_work_start >= 0
    AND days_work_start_to_completion >= 0
    AND days_completion_to_delivery >= 0
    AND is_completed = 1
GROUP BY store_id,store_name, store_type, manager_name, service_type
ORDER BY avg_total_cycle_time DESC

### Estimator Accuracy

In [0]:
%sql

WITH estimate_versions AS (
    SELECT 
        order_id,
        estimator_id,
        estimator_name,
        estimate_version_no,
        estimate_amount,
        invoice_amount,
    
        FIRST_VALUE(estimate_amount) OVER (
            PARTITION BY order_id 
            ORDER BY estimate_version_no ASC
        ) AS initial_estimate,
        
        -- Get final estimate (last version)
        LAST_VALUE(estimate_amount) OVER (
            PARTITION BY order_id 
            ORDER BY estimate_version_no DESC
        ) AS final_estimate,
        
        ROW_NUMBER() OVER (
            PARTITION BY order_id 
            ORDER BY estimate_version_no DESC
        ) AS is_latest
        
    FROM automobilerepair.gold.cube_kpi_metrics
    WHERE estimator_id IS NOT NULL 
      AND estimate_amount IS NOT NULL 
      AND invoice_amount IS NOT NULL 
      AND estimate_version_no IS NOT NULL
),

final_records AS (
    -- Keep only one record per order (the final version)
    SELECT 
        estimator_id,
        estimator_name,
        order_id,
        initial_estimate,
        final_estimate,
        invoice_amount
    FROM estimate_versions
    WHERE is_latest = 1
),

estimator_metrics AS (
    SELECT 
        estimator_id,
        estimator_name,
        COUNT(DISTINCT order_id) AS total_orders,
        
        -- Average variance from final estimate to actual invoice (in percentage)
        ROUND(
            AVG(
                CASE 
                    WHEN invoice_amount > 0 THEN ABS(final_estimate - invoice_amount) * 100.0 / invoice_amount
                    ELSE NULL 
                END
            ), 2
        ) AS avg_variance_pct,
        
        ROUND(AVG(final_estimate - initial_estimate), 2) AS avg_estimate_adjustment
        
    FROM final_records
    GROUP BY estimator_id, estimator_name
)

SELECT 
    estimator_name,
    estimator_id,
    total_orders,
    avg_estimate_adjustment,
    avg_variance_pct,
    RANK() OVER (ORDER BY avg_variance_pct ASC) AS accuracy_rank
FROM estimator_metrics
ORDER BY accuracy_rank ASC;

### Technician Workload

In [0]:
%sql
WITH technician_workload AS (
    SELECT
        technician_id,
        technician_name,
        invoice_year_month,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(days_work_start_to_completion) AS total_working_days

    FROM automobilerepair.gold.cube_kpi_metrics
    WHERE is_completed = 1
    GROUP BY 
        technician_id,
        technician_name,
        invoice_year_month
),

ranked_technicians AS (
    SELECT *,
        RANK() OVER ( PARTITION BY invoice_year_month ORDER BY total_working_days DESC) AS workload_rank FROM technician_workload
)
SELECT *
FROM ranked_technicians
ORDER BY invoice_year_month, workload_rank;